# Modules 4–6: Evaluate & Review, Monitor & Alert, Bring Your Own Agents

> Part 2 of the workshop, ~25 min. Prerequisite: Modules 1–2 (the agent is registered and has traces in development).

| Module | What you'll do |
|---|---|
| **4. Evaluate and review** (~10 min) | Save runs to a dataset, score them with "are the themes backed by quoted responses?" (a deterministic check plus an LLM-as-judge), run it **online** on new traces, and route low scores to an annotation queue for survey owners. |
| **5. Monitor and alert** (~10 min) | Read the prebuilt dashboard, build a custom one, and set alerts on error rate, latency, and a drop in evaluator score. |
| **6. Bring your own agents** (~5 min) | Trace a **Google ADK** agent into the same instance with LangSmith's ADK integration, score it with the **same evaluators**, and see it next to the Deep Agent in the Agents view. |

## Setup

In [ ]:
import os, sys, time
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)

# Agent addressing and LANGSMITH_PROJECT can't be combined: with both set, the SDK
# can't pick a destination and drops the trace.
if os.environ.pop("LANGSMITH_PROJECT", None):
    print("Note: ignoring LANGSMITH_PROJECT in this kernel (conflicts with agent addressing).")

import warnings
# Keep outputs readable: agent addressing is in beta (one-time notice), and
# list_runs() is deprecated but supported until 2027.
warnings.filterwarnings("ignore", message="Addressing runs with `langsmith.address` is in beta")
warnings.filterwarnings("ignore", message=r"list_runs\(\) is deprecated")

import langsmith as ls
from langsmith import Client, uuid7

from utils.models import model
from utils.agent_registry import ensure_agent, environment_project_name

AGENT_ID = "survey-feedback-agent"   # the agent's identifier in LangSmith (lowercase, hyphens)
ENVIRONMENT = "development"
dev_address = ls.agent_address(AGENT_ID, ENVIRONMENT)

client = Client()
# Models go through the LangSmith LLM Gateway. `langsmith:<provider>/<model>` models
# read LANGSMITH_GATEWAY_API_KEY (then LANGSMITH_API_KEY); reuse the workshop's gateway key.
if os.environ.get("LANGSMITH_API_KEY_GATEWAY"):
    os.environ.setdefault("LANGSMITH_GATEWAY_API_KEY", os.environ["LANGSMITH_API_KEY_GATEWAY"])
for key in ["LANGSMITH_API_KEY", "LANGSMITH_API_KEY_GATEWAY"]:
    print(f"{key:26s} set: {bool(os.environ.get(key))}")

from langgraph.checkpoint.memory import MemorySaver
from agents.local_agent import build_agent
from utils.langsmith_rules import get_or_create_annotation_queue, create_run_rule, delete_run_rule

ensure_agent(client, AGENT_ID, name="Survey Feedback Agent")
dev_project = environment_project_name(client, AGENT_ID, ENVIRONMENT)


def ask(question: str, *, traced_to=dev_address) -> dict:
    """Run the survey agent once. It never posts: the run stops at the approval
    gate and returns the message that *would* be posted."""
    agent = build_agent(interrupts=True, checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": str(uuid7())}}
    inputs = {"messages": [{"role": "user", "content": question}]}
    if traced_to is None:
        result = agent.invoke(inputs, config=config)
    else:
        with ls.tracing_context(address=traced_to, client=client):
            result = agent.invoke(inputs, config=config)
    if result.get("__interrupt__"):
        action = result["__interrupt__"][0].value["action_requests"][0]
        return {"answer": action["args"]["message"], "pending_post": True}
    return {"answer": result["messages"][-1].text, "pending_post": False}

---
# Module 4. Evaluate and Review

### 4.1 Generate a few runs

If Modules 1–3 already left traces in development, this tops them up with a spread of typical requests.

In [ ]:
PROMPTS = [
    "Summarize the top themes in the latest survey export for the survey owners.",
    "What are customers saying about curbside and in-store pickup? Use quotes.",
    "Which product categories have the worst NPS, and why? Back each with quotes.",
    "Give me a 3-theme summary of the appliance delivery feedback.",
    "Draft a summary of what is working well, with quotes, and post it to #store-ops.",
]
for q in PROMPTS:
    out = ask(q)
    print(f"- {q[:60]}... -> {len(out['answer'])} chars{' (stopped at approval)' if out['pending_post'] else ''}")
client.flush()

### 4.2 Save runs to a dataset

Production runs are the best source of test cases. Pull recent root runs from the development environment, keep each one's user request, and save it as a dataset example. In the UI you'd do the same from a trace with **Add to dataset**, which is how a weak output found in a trace becomes a regression test.

In [ ]:
time.sleep(5)


def user_request(inputs: dict) -> str | None:
    """Pull the user's request out of a traced agent input."""
    for m in (inputs or {}).get("messages", []):
        if isinstance(m, dict):
            kw = m.get("kwargs", {})
            if m.get("role") in ("user", "human") or m.get("type") == "human":
                return m.get("content")
            if kw.get("type") == "human":
                return kw.get("content")
    return None


DATASET = "survey-feedback-agent-runs"
dataset = (client.read_dataset(dataset_name=DATASET) if client.has_dataset(dataset_name=DATASET)
           else client.create_dataset(DATASET, description="Survey-agent requests saved from development traces."))

existing = {ex.inputs.get("question") for ex in client.list_examples(dataset_id=dataset.id)}
new_examples = []
for run in client.list_runs(project_name=dev_project, is_root=True, error=False, limit=50):
    q = user_request(run.inputs)
    if q and q not in existing:
        existing.add(q)
        new_examples.append({"inputs": {"question": q}, "metadata": {"source_run_id": str(run.id)}})
if new_examples:
    client.create_examples(dataset_id=dataset.id, examples=new_examples)
print(f"{DATASET}: {len(existing)} examples ({len(new_examples)} added)")

### 4.3 The evaluators: are the themes backed by quoted responses?

Two complementary checks:

- **`quotes_verbatim`** (code, deterministic): every quote cited as `"..." (R-xxxx)` must appear word for word in that response's comment. This catches invented or paraphrased quotes, which an LLM judge can miss.
- **`themes_grounded`** (LLM-as-judge): every theme is supported by at least two quoted responses that actually say what the theme claims.

In [ ]:
import re
from typing_extensions import Annotated, TypedDict
from survey_agent.tools.survey import load_survey_export

COMMENTS = {r["response_id"]: r["comment"] for r in load_survey_export()}
QUOTE = re.compile(r'"([^"]{12,}?)"\s*\(?\s*(R-\d{4})')


def _norm(s: str) -> str:
    return " ".join(s.lower().replace("\u2019", "'").split())


def _quote_is_verbatim(quote: str, rid: str) -> bool:
    source = _norm(COMMENTS.get(rid, ""))
    parts = [p.strip(" .,") for p in re.split(r"\[redacted\]|\.\.\.|\u2026", _norm(quote))]
    return bool(source) and all(p in source for p in parts if p)


def quotes_verbatim(outputs: dict) -> dict:
    text = outputs.get("answer", "").replace("\u201c", '"').replace("\u201d", '"')
    quotes = QUOTE.findall(text)
    if not quotes:
        return {"key": "quotes_verbatim", "score": 0, "comment": "No quotes cited with a response ID."}
    ok = [_quote_is_verbatim(q, rid) for q, rid in quotes]
    return {"key": "quotes_verbatim", "score": sum(ok) / len(ok),
            "comment": f"{sum(ok)}/{len(ok)} quotes found verbatim in the cited response."}


class Grade(TypedDict):
    reasoning: Annotated[str, ..., "One or two sentences explaining the grade."]
    grounded: Annotated[bool, ..., "True if every theme is backed by at least two quoted responses that support it."]


JUDGE_PROMPT = (
    "You review customer-survey theme summaries written for survey owners. A summary is "
    "grounded only if EVERY theme it names is backed by at least two quoted customer responses, "
    "and each quote actually supports the theme it is listed under. Themes with no quotes, one "
    "quote, or quotes about something else make the summary ungrounded."
)
judge = model.with_structured_output(Grade)


def themes_grounded(inputs: dict, outputs: dict) -> dict:
    grade = judge.invoke([
        ("system", JUDGE_PROMPT),
        ("user", f"Request: {inputs['question']}\n\nSummary:\n{outputs['answer']}"),
    ])
    return {"key": "themes_grounded", "score": int(grade["grounded"]), "comment": grade["reasoning"]}


print(quotes_verbatim({"answer": '"Delivery window for my washer was 8am-5pm" (R-1003)'}))
print(quotes_verbatim({"answer": '"Delivery was always late and rude" (R-1003)'}))

### 4.4 Run an offline experiment

`client.evaluate` runs the agent on every example and attaches both scores. Experiments get their own destination, separate from the agent's environments, so test traffic never mixes with real traces.

In [ ]:
baseline = client.evaluate(
    lambda inputs: ask(inputs["question"], traced_to=None),
    data=DATASET,
    evaluators=[quotes_verbatim, themes_grounded],
    experiment_prefix="survey-agent-baseline",
    max_concurrency=2,
)
print("Experiment:", baseline.experiment_name)

**Improve and re-run.** Open the experiment (**Datasets & Experiments** → `survey-feedback-agent-runs`) and sort by `quotes_verbatim`. Find a weak output, then change the prompt or skill:

- `survey_agent/instructions.md`: the workflow and rules, e.g. "copy quotes character-for-character from `search_responses` results".
- `survey_agent/skills/survey-summary/SKILL.md`: the output format.

Re-run the cell above with `experiment_prefix="survey-agent-v2"` and compare the two experiments side by side to see whether the score moved. `ask()` compiles `survey_agent/` fresh on every call, so the re-run picks up your edit. Ship it with `mda deploy` (Module 3). Because instructions and skills live in Context Hub, you can also edit them there for an immediate change without a redeploy; the next `mda deploy` asks before overwriting Hub edits.

### 4.5 Online LLM-as-judge on new traces

Offline experiments test changes before they ship; **online evaluators** score live traffic as it arrives. This rule runs the grounding judge on every new root run in the development environment.

> The judge runs on LangSmith's infrastructure, so it needs a model key stored as a workspace secret (**Settings → Secrets**, e.g. `OPENAI_API_KEY`).

In [ ]:
grounded_schema = {
    "title": "themes_grounded",
    "description": "Whether every theme in the summary is backed by quoted survey responses.",
    "type": "object",
    "properties": {
        "themes_grounded": {"type": "boolean", "description": "True if every theme has at least two supporting quotes."},
        "comment": {"type": "string", "description": "One short sentence explaining the score."},
    },
    "required": ["themes_grounded", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=dev_project,
    display_name="survey-online-themes-grounded",
    sampling_rate=1.0,
    filter='and(eq(is_root, true), neq(name, "summarize_one_response"))',
    llm_judge_prompt=JUDGE_PROMPT + " Score the agent's final output.",
    llm_judge_schema=grounded_schema,
)
print("Online evaluator:", online_rule["url"])

### 4.6 Route low scores to survey owners

Runs the judge marks ungrounded go to an **annotation queue**, where survey owners review them, correct the summary, and add the good version back to the dataset.

In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="survey-summaries-needs-review",
    description="Survey summaries whose themes were not backed by quoted responses.",
)
queue_rule = create_run_rule(
    client,
    project_name=dev_project,
    display_name="survey-route-ungrounded",
    filter=(
        'and(eq(is_root, true), eq(feedback_key, "themes_grounded"), lt(feedback_score, 0.5))'
    ),
    add_to_annotation_queue_id=queue.id,
)
print("Queue rule:", queue_rule["url"])
print(f"Queue:      https://smith.langchain.com/o/{queue_rule['payload']['tenant_id']}/annotation-queues/{queue.id}")

In [ ]:
# Rules apply to runs that arrive after they're created. Give them a minute, then trigger a few.
time.sleep(60)
for q in PROMPTS[:3]:
    ask(q)
client.flush()
print("Scores appear on these traces in about a minute; low scores then land in the queue.")

---
# Module 5. Monitor and Alert

### 5.1 Dashboards

- **Prebuilt**: open the agent → **Monitoring** → **Environments** → **Development**. Trace count, error rate, latency, tokens and cost, and feedback scores (including `themes_grounded`) are generated automatically for each environment.
- **Custom**: **Monitoring** → **Custom** → **+ Dashboard** named "Survey agent health". Add charts:
  1. **Error rate**: runs, metric *error rate*, filter root runs.
  2. **Latency**: runs, metric *P50 and P99 latency*, filter root runs.
  3. **Grounding score**: feedback, metric *average* `themes_grounded`.
  4. **Approvals**: tool runs where name is `post_themes_to_channel`, count over time.

  Tag the dashboard with the agent's resource tag so it shows under the agent's **Custom** tab.

Use the baseline below to pick alert thresholds.

In [ ]:
from datetime import datetime, timedelta, timezone
from statistics import mean

since = datetime.now(timezone.utc) - timedelta(hours=24)
roots = list(client.list_runs(project_name=dev_project, is_root=True, start_time=since,
                              filter='neq(name, "summarize_one_response")'))
latencies = sorted((r.end_time - r.start_time).total_seconds() for r in roots if r.end_time)
error_rate = sum(bool(r.error) for r in roots) / max(len(roots), 1)
pct = lambda q: latencies[min(len(latencies) - 1, int(q * len(latencies)))] if latencies else float("nan")
scores = [f.score for f in client.list_feedback(run_ids=[r.id for r in roots], feedback_key=["themes_grounded"])
          if f.score is not None]

print(f"Root runs (24h):     {len(roots)}")
print(f"Error rate:          {error_rate:.0%}")
print(f"Latency p50 / p95:   {pct(0.5):.1f}s / {pct(0.95):.1f}s")
print(f"themes_grounded avg: {mean(scores):.2f} over {len(scores)} scored runs" if scores else "themes_grounded: no scores yet")

### 5.2 Alerts

Alerts are scoped to one environment of one agent, so production alerts ignore dev and staging traffic without any filters. Open the agent → **Tracing** → choose the environment → **Alerts** (top right) → **+ New alert**. Create three alerts:

| Alert | Metric | Condition (starting point) | Window |
|---|---|---|---|
| Error rate | **Errors**, percent | ≥ 5% | 5 min |
| Latency | **Latency**, average | ≥ ~1.5× the p95 above | 15 min |
| Grounding drop | **Feedback Score**, key `themes_grounded`, average | ≤ 0.7 | 15 min |

Use **Preview** to see how often each threshold would have fired on recent data, then route notifications to **Slack**, **PagerDuty**, or a **webhook** (the webhook tab has a Microsoft Teams recipe). Create them on **Development** today, and repeat on **Production** when you go live.

---
# Module 6. Bring Your Own Agents

Agents built outside LangSmith trace into the same instance. Here's the Google ADK version of the survey agent (`agents/adk_agent.py`): same tools, same task, running in this process instead of as a Managed Deep Agent. `configure_google_adk()` instruments ADK (runner, agents, model calls, tools) and sends the traces to LangSmith, addressed to the ADK agent's own environment. LangSmith also accepts OpenTelemetry (OTLP) from any framework; see 6.3.

> The ADK agent uses the same model as the Managed Deep Agent, `gpt-5.6-luna` through the LangSmith LLM Gateway, via ADK's LiteLLM wrapper. No Google key is needed. Set `ADK_MODEL` to another Gateway model ID to compare models, or to a `gemini-*` name to call Gemini natively.

In [ ]:
ADK_AGENT_ID = os.environ.get("ADK_AGENT_ID", "survey-feedback-adk")
ensure_agent(client, ADK_AGENT_ID, name="Survey Feedback Agent (Google ADK)")

from agents.adk_agent import enable_langsmith_tracing, run_adk_agent

from agents.adk_agent import ADK_MODEL
print("ADK model:", ADK_MODEL, "(LangSmith LLM Gateway)" if not ADK_MODEL.startswith("gemini") else "(Gemini, native)")

# Route this process's ADK traces to the ADK agent's development environment.
enable_langsmith_tracing(ADK_AGENT_ID, "development")

answer = await run_adk_agent(
    "Summarize the top survey themes for the survey owners, with two verbatim quotes per theme."
)
print(answer)

Select **Survey Feedback Agent (Google ADK)** → **Tracing** → **Development**. The trace shows the ADK agent invocation, the transfer to `theme_clustering_agent`, each tool call, and the model calls to `gpt-5.6-luna` through the Gateway.

### 6.1 Same evaluators, different framework

Run the **same dataset** through the ADK agent with the **same evaluators**. Because the score definitions don't depend on the framework, the two experiments compare directly.

In [ ]:
from langsmith import aevaluate


async def adk_target(inputs: dict) -> dict:
    return {"answer": await run_adk_agent(inputs["question"])}


adk_results = await aevaluate(
    adk_target,
    data=DATASET,
    evaluators=[quotes_verbatim, themes_grounded],
    experiment_prefix="survey-agent-adk",
    max_concurrency=2,
    client=client,
)
print("Experiment:", adk_results.experiment_name)

In **Datasets & Experiments** → `survey-feedback-agent-runs`, select `survey-agent-baseline` and `survey-agent-adk` → **Compare** to see the Deep Agent and the ADK agent scored side by side.

### 6.2 The Agents view

Open **Agents**. The Fleet agent from the demo, the Deep Agent, and the ADK agent are listed together, each with its environments, traces and evaluators. This is the starting point for an agent registry.

### 6.3 Connect one of your own

Register the agent once (UI: **+ Agent** → **Observability**, or `ensure_agent(client, "<id>", "<name>")`), then send traces to it:

```python
# Any Python code
import langsmith as ls

@ls.traceable(address=ls.agent_address("<your-agent-id>", "development"))
def handle(request): ...
```

```bash
# LangChain / LangGraph / ADK / other SDK integrations: environment variables
export LANGSMITH_TRACING=true
export LANGSMITH_AGENT_ID=<your-agent-id>
export LANGSMITH_AGENT_ENVIRONMENT=development   # don't also set LANGSMITH_PROJECT
```

For other languages, send OTLP to LangSmith with the `Langsmith-Agent-Id` and `Langsmith-Agent-Environment` headers.

---
## Part 7. Engine — Automated Agent Improvement (tying it together)

We've now traced the feedback agent, evaluated it offline and online, routed flagged runs to a queue, and weighed model choice. **Engine ties all of that into one continuous loop.**

Querying traces by hand (Part 2) works because the project is small. Production agents push past that quickly:

- **Agents are longer-running** — a single trace can span dozens of tool calls and multiple subagent invocations.
- **Multimodal inputs and outputs** — traces include images, audio, attachments, and structured documents alongside text.
- **Agents are proliferating** — a typical deployment runs many agents, each producing its own trace stream.

**Engine performs this trace inspection automatically and continuously.** It turns the trace corpus into a continuous-improvement workflow: it surfaces recurring issues, diagnoses their root cause, and guides you through fixing them and preventing them from coming back. (Docs: <https://docs.langchain.com/langsmith/engine>.)


### 7.1 What Engine does

Engine plugs into the full agent engineering lifecycle: **trace → recurring failure detected → root cause diagnosed → fix proposed → evaluator deployed → dataset examples generated.** It runs continuously against the project's traces (scanning every ~6 hours by default) and surfaces issues without requiring manual review.

Each issue ships with a proposed code or prompt fix you can apply as a pull request, a suggested evaluator to catch regressions on the same failure mode, and dataset examples with assertions you can promote into experiments — closing the loop back into your regression suite.


### 7.2 Enabling Engine

Two configuration steps:

1. **Organization Admin** enables Engine workspace-wide in org settings.
2. **Per project**, configure:
   - Optionally connect a **GitHub repository** so fix proposals can become pull requests directly.
   - Select **priority categories** — which failure types matter most.
   - Review the **auto-generated agent overview** document Engine produces; correct any misunderstandings before issue generation starts.

Once enabled, Engine populates an **Issues** list in the project as it identifies failure patterns.

<img src="../images/engine_config.png" alt="Engine configuration dialog with GitHub connection and priority categories" style="width: auto; max-height: 380px; border-radius: 8px;">


### 7.3 The closed loop

Each Engine issue includes four components:

- **Diagnosis** — the root cause, linked to the failing traces. Inspect the linked traces directly to confirm the diagnosis matches your read.
- **Proposed Fix** — code or prompt changes; openable as a PR with one click if a GitHub repo is connected.
- **Suggested Evaluator** — a check you can deploy as a new run rule to catch regressions on this failure mode going forward.
- **Add offline examples** — extract the failing runs as dataset entries for the experiment loop.

<img src="../images/engine_triage.png" alt="Engine triage view with the issues list" style="width: auto; max-height: 420px; border-radius: 8px;">

<img src="../images/engine_issue.png" alt="Engine issue detail with diagnosis, proposed fix, and suggested evaluator" style="width: auto; max-height: 420px; border-radius: 8px;">

Priority adjustments and feedback (accept / reject / refine) train Engine's understanding of which issues matter for this specific project — relevance improves over iterations.

**Engine and the rest of this notebook close the loop together:**

- **Suggested evaluators → Part 4** (online run rules — same `create_run_rule` helper).
- **Dataset examples with assertions → Part 3** (offline experiments — promote them directly onto a dataset).
- **Issues that need human review → Part 5** (annotation queues — same routing rule pattern).


## Recap

- **Evaluate and review**: runs were saved to a dataset, scored offline and online for quote-backed themes, and low scores were routed to survey owners.
- **Monitor and alert**: per-environment dashboards, plus alerts on error rate, latency, and grounding score.
- **Bring your own agents**: an ADK agent traced with LangSmith's ADK integration, scored by the same evaluators, and listed in the same Agents view.

To clean up the rules from this notebook: `delete_run_rule(client, online_rule["id"])` and `delete_run_rule(client, queue_rule["id"])`.